# 00. 직무 역량을 연구 커리큘럼으로 바꾸기

목표는 채용 공고형 요구사항을 개인 학습 로드맵, 실험 체크리스트, 논문 산출물로 바꾸는 것입니다.
회사 이름이나 특정 조직 이름이 필요한 곳은 `xxx`로 표기합니다.

최종 연구 주제:

**노이즈가 있는 현장 상황에서 ASR, 환경음, OCR 증거를 결합해 문서/상황 질문응답의 신뢰도를 높이는 경량 멀티모달 파이프라인**

이 주제는 음성 인식, 음성 대화, 환경음 이해, OCR, VLM, 대규모 데이터 전처리, 평가, MLOps, 논문 작성까지 연결됩니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
job_requirements = [
    ("음성 인식", "ASR 전처리, WER/CER, 노이즈 강건성, 오류 분석"),
    ("음성 대화", "의도 분류, 슬롯 추출, 대화 상태, 응답 평가"),
    ("환경음 이해", "오디오 이벤트 분류, 임베딩, 시간 구간 추론"),
    ("이미지 이해", "이미지 전처리, OCR, 레이아웃 인식, TextVQA"),
    ("대규모 데이터", "수집 설계, 품질 지표, PII/회사명 마스킹, 누수 방지"),
    ("연구 재현", "논문 읽기, 베이스라인, ablation, 신뢰구간"),
    ("MLOps", "실험 추적, 모델 카드, 데이터 카드, 배포 전 체크"),
    ("기술 리드", "코드 리뷰, 의사결정 로그, 멘토링 자료화"),
    ("논문/대회", "문헌 매트릭스, 실험표, 그림, 논문 초안"),
]

roadmap = pd.DataFrame(job_requirements, columns=["요구 역량", "학습/실습 포인트"])
roadmap["노트북"] = [
    "01_audio_speech_language/01_audio_feature_baselines.ipynb",
    "03_multimodal_dialogue/01_speech_dialogue_state_machine.ipynb",
    "01_audio_speech_language/03_environmental_sound_understanding.ipynb",
    "02_vision_ocr_language/01_ocr_robustness_lab.ipynb",
    "04_data_mlops_leadership/01_data_engineering_quality.ipynb",
    "05_research_paper_project/02_reproduction_and_ablation.ipynb",
    "04_data_mlops_leadership/02_experiment_tracking_and_model_cards.ipynb",
    "04_data_mlops_leadership/03_code_review_mentoring_playbook.ipynb",
    "05_research_paper_project/03_paper_figures_and_draft.ipynb",
]
roadmap.to_csv(ARTIFACTS / "skill_to_notebook_map.csv", index=False, encoding="utf-8-sig")
roadmap


In [ ]:
def mask_company_names(text: str, extra_terms=None) -> str:
    """민감한 회사명/조직명 후보를 xxx로 바꾸는 간단한 전처리기입니다.

    실제 프로젝트에서는 사내 금칙어 사전, 정규표현식, NER 결과를 함께 사용합니다.
    여기서는 공개 예제이므로 사용자가 직접 extra_terms에 민감어를 넣도록 설계합니다.
    """
    extra_terms = extra_terms or []
    masked = text
    patterns = [
        r"회사명\s*[:：]\s*[^\n,;]+",
        r"소속\s*[:：]\s*[^\n,;]+",
        r"client\s*[:：]\s*[^\n,;]+",
        r"vendor\s*[:：]\s*[^\n,;]+",
    ]
    import re
    for pat in patterns:
        masked = re.sub(pat, lambda m: m.group(0).split(":")[0] + ": xxx", masked, flags=re.IGNORECASE)
    for term in extra_terms:
        if term:
            masked = masked.replace(term, "xxx")
    return masked

sample = "회사명: ExampleCorp, 음성 OCR 모델 개발 리드 / vendor: SecretVendor"
mask_company_names(sample, extra_terms=["ExampleCorp", "SecretVendor"])


## 완료 기준

- `artifacts/skill_to_notebook_map.csv`가 생성된다.
- 본인의 경력/부족한 역량을 노트북 경로와 연결한다.
- 최종 논문 주제가 너무 넓다고 느껴지면 `음성+OCR`, `환경음+OCR`, `문서VQA` 중 하나로 축소한다.
